# Remote Sensing CNN + Transformer Pipeline

This notebook is the main Colab-friendly runner. It uses the same scripts as the command line project, so notebook experiments stay reproducible.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
CONFIG = ROOT / 'configs' / 'preprocessing.json'
print(ROOT)
print(CONFIG.read_text())

## 1. Build Or Reuse Clean Patch Dataset

If `downloadedRawData/` exists, this rebuilds the NPZ from raw TIFFs. If raw data is not present, it reuses the committed compact NPZ artifact.

In [ ]:
npz_path = ROOT / 'artifacts' / 'patches_clean' / 'train_cnn_transformer_15x15.npz'
raw_data_dir = ROOT / 'downloadedRawData'

if raw_data_dir.exists():
    cmd = [sys.executable, 'scripts/preprocess_data.py', '--config', str(CONFIG)]
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(result.returncode)
elif npz_path.exists():
    print('Raw downloadedRawData/ folder is not present. Reusing existing compact NPZ:', npz_path)
else:
    raise FileNotFoundError('No raw downloadedRawData/ folder and no compact NPZ artifact found.')

## 2. Inspect Dataset Schema

In [ ]:
import numpy as np
npz_path = ROOT / 'artifacts' / 'patches_clean' / 'train_cnn_transformer_15x15.npz'
data = np.load(npz_path, allow_pickle=False)
print(data.files)
for key in ['patches', 'valid_pixel_mask', 'band_mask', 'time_mask', 'time_doy', 'phenophase_doy']:
    if key in data.files:
        print(key, data[key].shape, data[key].dtype)
print('bands:', data['bands'])

## 3. View Report Charts

These charts are regenerated every preprocessing run. They should be good enough for sanity checking and article notes.

In [ ]:
from IPython.display import display, Image
manifest_path = ROOT / 'artifacts' / 'preprocessing_report' / 'chart_manifest.json'
manifest = json.loads(manifest_path.read_text())
for rel in manifest['charts']:
    print(rel)
    display(Image(filename=str(ROOT / 'artifacts' / 'preprocessing_report' / rel)))

## 4. View Deterministic Patch Samples

Samples are grouped as `random`, `invalid`, and `edge`, so they are not accidentally dominated by only broken cells.

In [ ]:
for rel in manifest['samples']:
    if rel.endswith('.png'):
        print(rel)
        display(Image(filename=str(ROOT / 'artifacts' / 'preprocessing_report' / rel)))

## 5. Train CNN + Transformer

For a quick smoke test use `configs/train_baseline.json`. For a good Colab GPU, use `configs/train_colab_gpu.json`. Start with 1 epoch to verify, then run the full config.

In [ ]:
try:
    import torch
    print('torch', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    cmd = [sys.executable, 'scripts/train_baseline.py', '--config', 'configs/train_colab_gpu.json', '--epochs', '1']
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except ImportError:
    print('PyTorch is not installed. In Colab run: !pip install -r requirements.txt')

## 6. Submission-Time Shape

Final submission should run test extraction from `/input` and prediction to `/output/result.json`. The exact result JSON schema still needs the official guide restored/confirmed; only `inference/write_submission.py` should need changing if the schema differs.

In [ ]:
# Example for the final environment, not for local training:
# !python scripts/extract_test_patches.py --input-root /input --output-npz /workspace/patches_clean_test/test_cnn_transformer_15x15.npz
# !python scripts/run_inference.py --checkpoint artifacts/models/cnn_transformer_baseline/model.pt --output-json /output/result.json